# 03 - Ingeniería de Prompts

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2-2.5 horas

---

## Índice

1. [Introducción a la Ingeniería de Prompts](#intro)
2. [Zero-Shot Prompting](#zeroshot)
3. [Few-Shot Prompting](#fewshot)
4. [Prompts con Rol y Contexto](#rol)
5. [Restricciones y Formato de Salida](#formato)
6. [Chain of Thought (CoT)](#cot)
7. [Temperatura y Diversidad](#temperatura)
8. [Mejores Prácticas](#mejores)
9. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Diseñar prompts efectivos para diferentes tareas
- Aplicar técnicas como zero-shot, few-shot y chain of thought
- Controlar el formato y estilo de las respuestas
- Entender y ajustar parámetros como temperatura
- Evitar errores comunes en prompt engineering

<a name="intro"></a>
## 1. Introducción a la Ingeniería de Prompts

### ¿Qué es la Ingeniería de Prompts?

La **ingeniería de prompts** (prompt engineering) es el arte y ciencia de diseñar instrucciones efectivas para modelos de lenguaje. Un buen prompt puede marcar la diferencia entre una respuesta útil y una inservible.

### Anatomía de un prompt

Un prompt efectivo típicamente incluye:

1. **Instrucción**: Qué quieres que haga el modelo
2. **Contexto**: Información de fondo relevante
3. **Entrada**: Los datos específicos a procesar
4. **Formato de salida**: Cómo quieres la respuesta

```
┌─────────────────────────────────────┐
│  [ROL/SISTEMA]                      │
│  Eres un experto en...              │
├─────────────────────────────────────┤
│  [CONTEXTO]                         │
│  Dado el siguiente documento...     │
├─────────────────────────────────────┤
│  [INSTRUCCIÓN]                      │
│  Resume en 3 puntos principales...  │
├─────────────────────────────────────┤
│  [FORMATO]                          │
│  Responde en formato JSON...        │
└─────────────────────────────────────┘
```

## Configuración

Para este notebook usaremos **Groq**, que ofrece un tier gratuito generoso con modelos como Llama 3.3.

Obtén tu API key gratis en: https://console.groq.com/keys

In [1]:
# Install required libraries
%pip install -U langchain langchain-groq python-dotenv

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

# Get API key (Groq is FREE with generous limits)
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key (gratis en console.groq.com): ")

print("API Key configurada ✓")

API Key configurada ✓


In [3]:
from langchain_groq import ChatGroq

# Initialize the LLM (using free Groq API)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0  # Start with deterministic responses
)

def get_completion(prompt, temperature=0):
    """Get completion from LLM with specified temperature."""
    llm_temp = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=temperature)
    response = llm_temp.invoke(prompt)
    return response.content

# Test the connection
test_response = get_completion("Di 'Hola, el sistema funciona correctamente'")
print(f"Test: {test_response}")

c:\Users\ignar\iniciacion_NLP_LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Test: ¡Hola! Me alegra saber que el sistema funciona correctamente. ¿En qué puedo ayudarte hoy? ¿Necesitas ayuda con algo en particular o simplemente quieres charlar un rato? Estoy aquí para ayudarte en lo que necesites.


<a name="zeroshot"></a>
## 2. Zero-Shot Prompting

**Zero-shot** significa que el modelo realiza la tarea sin ejemplos previos. Solo recibe la instrucción.

### El problema del prompt ambiguo

In [4]:
# Ambiguous prompt
prompt_ambiguo = "Describe un animal."

response = get_completion(prompt_ambiguo)
print("Prompt ambiguo:")
print(f"'{prompt_ambiguo}'")
print(f"\nRespuesta:\n{response}")

Prompt ambiguo:
'Describe un animal.'

Respuesta:
¡Claro! Me gustaría describir un animal fascinante: el **panda gigante**.

**Características físicas:**

* El panda gigante es un mamífero grande y robusto, con un peso que puede alcanzar hasta 160 kg.
* Su pelaje es grueso y suave, con un color blanco y negro característico que le ayuda a camuflarse en su hábitat.
* Tiene una cabeza grande y redonda, con ojos pequeños y orejas redondas.
* Sus patas son cortas y fuertes, con garras grandes y afiladas que le permiten trepar y agarrarse a los árboles.

**Hábitat y comportamiento:**

* El panda gigante se encuentra en las montañas de China, en bosques de bambú y coníferas.
* Es un animal herbívoro, y se alimenta principalmente de bambú, que constituye hasta el 99% de su dieta.
* Es un animal solitario y territorial, y cada individuo tiene su propio territorio que marca con su olor.
* El panda gigante es un animal tranquilo y pacífico, y no es agresivo a menos que se sienta amenazado o esté

In [5]:
# Specific prompt
prompt_especifico = """Describe las características principales de un guepardo, incluyendo:
- Velocidad máxima
- Hábitat natural
- Dieta

Responde en no más de 50 palabras."""

response = get_completion(prompt_especifico)
print("Prompt específico:")
print(f"'{prompt_especifico}'")
print(f"\nRespuesta:\n{response}")

Prompt específico:
'Describe las características principales de un guepardo, incluyendo:
- Velocidad máxima
- Hábitat natural
- Dieta

Responde en no más de 50 palabras.'

Respuesta:
El guepardo: velocidad máxima de 120 km/h, hábitat en sabanas y praderas, dieta de pequeños mamíferos y aves.


### Lecciones aprendidas

| Prompt ambiguo | Prompt específico |
|---------------|------------------|
| "Describe un animal" | "Describe las características de X" |
| Sin límites | "En no más de 50 palabras" |
| Sin estructura | Lista con puntos específicos |

<a name="fewshot"></a>
## 3. Few-Shot Prompting

**Few-shot** proporciona ejemplos para que el modelo entienda el patrón deseado.

### Clasificación de sentimiento

In [6]:
# Zero-shot classification
text_to_classify = "Este producto es una basura total, perdí mi dinero."

prompt_zeroshot = f"""Clasifica el sentimiento del siguiente texto como POSITIVO, NEGATIVO o NEUTRAL.

Texto: "{text_to_classify}"

Sentimiento:"""

response = get_completion(prompt_zeroshot)
print("Zero-shot:")
print(response)

Zero-shot:
El sentimiento del texto es: NEGATIVO.

La razón es que el texto utiliza palabras y frases con connotaciones negativas, como "basura total" y "perdí mi dinero", lo que indica una fuerte insatisfacción y descontento con el producto. Esto sugiere que el autor del texto tiene una opinión muy negativa sobre el producto y su experiencia con él.


In [7]:
# Few-shot classification (with examples)
prompt_fewshot = f"""Clasifica el sentimiento del texto como POSITIVO, NEGATIVO o NEUTRAL.

Ejemplos:
Texto: "¡Me encanta este restaurante! La comida es deliciosa."
Sentimiento: POSITIVO

Texto: "El servicio fue terrible, nunca volveré."
Sentimiento: NEGATIVO

Texto: "El producto llegó a tiempo y funciona."
Sentimiento: NEUTRAL

Ahora clasifica:
Texto: "{text_to_classify}"
Sentimiento:"""

response = get_completion(prompt_fewshot)
print("Few-shot:")
print(response)

Few-shot:
El sentimiento del texto es: NEGATIVO

La razón es que el texto utiliza palabras y frases con connotación negativa, como "basura total" y "perdí mi dinero", lo que indica una experiencia desagradable y una opinión muy negativa sobre el producto.


### Few-shot para extracción de información

In [8]:
# Few-shot for information extraction
prompt_extraction = """Extrae la información de contacto del texto en formato estructurado.

Ejemplo 1:
Texto: "Contacta con María García al 612345678 o escribe a maria.garcia@empresa.com"
Resultado:
- Nombre: María García
- Teléfono: 612345678
- Email: maria.garcia@empresa.com

Ejemplo 2:
Texto: "Para más info, llama a Juan (tel: 699887766)"
Resultado:
- Nombre: Juan
- Teléfono: 699887766
- Email: No disponible

Ahora extrae:
Texto: "Mi asesora es Ana López, puedes contactarla en ana.lopez@consultoria.es o al móvil 677889900"
Resultado:"""

response = get_completion(prompt_extraction)
print(response)

- Nombre: Ana López
- Teléfono: 677889900
- Email: ana.lopez@consultoria.es


<a name="rol"></a>
## 4. Prompts con Rol y Contexto

Asignar un **rol** al modelo puede mejorar significativamente la calidad y relevancia de las respuestas.

In [9]:
# Without role
prompt_sin_rol = "¿Cómo puedo mejorar mi estrategia de marketing digital?"

response = get_completion(prompt_sin_rol)
print("SIN ROL:")
print(response[:500] + "...")

SIN ROL:
**Mejorar tu estrategia de marketing digital: Consejos y recomendaciones**

En la era digital, tener una estrategia de marketing efectiva es crucial para alcanzar tus objetivos y superar a la competencia. A continuación, te presento algunos consejos y recomendaciones para mejorar tu estrategia de marketing digital:

### 1. **Define tus objetivos y público objetivo**

Antes de empezar a crear contenido o campañas publicitarias, es importante definir tus objetivos y público objetivo. ¿Qué deseas l...


In [10]:
# With role
prompt_con_rol = """Eres un consultor experto en marketing digital con 15 años de experiencia 
trabajando con startups tecnológicas. Has ayudado a más de 100 empresas a escalar 
su presencia online.

Tu cliente es una startup de software B2B con presupuesto limitado (500€/mes para marketing).

Pregunta: ¿Cómo puedo mejorar mi estrategia de marketing digital?

Proporciona 3 recomendaciones concretas y actionables, priorizadas por impacto/coste."""

response = get_completion(prompt_con_rol)
print("CON ROL:")
print(response)

CON ROL:
**Mejorar la estrategia de marketing digital con un presupuesto limitado**

Entiendo que como startup de software B2B con un presupuesto limitado de 500€/mes para marketing, es crucial maximizar el impacto de cada euro invertido. A continuación, te presento 3 recomendaciones concretas y actionables, priorizadas por impacto/coste, para mejorar tu estrategia de marketing digital:

**1. Optimización del sitio web y contenido existente (Coste: 0€ - 50€/mes)**
 * **Impacto:** Alto
 * **Acciones:**
 + Realiza una auditoría del sitio web para identificar oportunidades de mejora en la experiencia del usuario y la conversión.
 + Ajusta el contenido existente para que sea más relevante y atractivo para tu audiencia objetivo.
 + Asegúrate de que el sitio web sea móvil y tenga una velocidad de carga rápida.
 * **Razón:** La optimización del sitio web y el contenido existente es una forma rentable de mejorar la experiencia del usuario y aumentar la conversión sin necesidad de invertir en n

### Diferentes roles para diferentes tareas

In [11]:
# Technical role
roles = {
    "programador": """Eres un ingeniero de software senior especializado en Python y arquitectura de sistemas.
    Siempre priorizas código limpio, mantenible y bien documentado.""",
    
    "profesor": """Eres un profesor universitario experto en explicar conceptos complejos de forma simple.
    Usas analogías y ejemplos cotidianos para facilitar la comprensión.""",
    
    "abogado": """Eres un abogado especializado en protección de datos y GDPR.
    Respondes de forma precisa citando la normativa aplicable."""
}

pregunta = "Explica qué es un API REST"

for rol_name, rol_desc in roles.items():
    prompt = f"{rol_desc}\n\nPregunta: {pregunta}\n\nRespuesta (máximo 100 palabras):"
    response = get_completion(prompt)
    print(f"\n{'='*50}")
    print(f"ROL: {rol_name.upper()}")
    print(f"{'='*50}")
    print(response)


ROL: PROGRAMADOR
Un API REST (Representational State of Resource) es un estilo de arquitectura para diseñar APIs. Se basa en el intercambio de recursos a través de HTTP, utilizando métodos como GET, POST, PUT y DELETE para interactuar con ellos. Permite una comunicación flexible y escalable entre sistemas, facilitando la integración de aplicaciones y servicios.

ROL: PROFESOR
Imagina un restaurante. Un API REST es como un mesero que te lleva pedidos al chef y te trae la comida lista. En lugar de comida, el mesero (API) lleva solicitudes de datos al "chef" (servidor) y te devuelve la respuesta. Utiliza métodos como "pedir" (GET), "entregar" (POST), "modificar" (PUT) y "cancelar" (DELETE) para interactuar con el servidor. De esta forma, puedes obtener o enviar datos de manera eficiente y organizada.

ROL: ABOGADO
Un API REST (Representational State of Resource) es un estilo de arquitectura de API que se basa en el protocolo HTTP. Según el Reglamento (UE) 2016/679, no se menciona específ

<a name="formato"></a>
## 5. Restricciones y Formato de Salida

Especificar el formato de salida es crucial para integrar respuestas de LLM en aplicaciones.

In [13]:
# JSON output format
prompt_json = """Analiza el siguiente texto y extrae la información en formato JSON.

Texto: "El evento tendrá lugar el 15 de marzo de 2024 en el Centro de Convenciones de Madrid. 
El precio de entrada es de 50 euros y habrá descuento del 20% para estudiantes."

Responde SOLO con el JSON, sin explicaciones adicionales.
Formato esperado:
{
    "fecha": "...",
    "lugar": "...",
    "precio": ...,
    "descuento_estudiantes": "..."
}"""

response = get_completion(prompt_json)
print("Salida JSON:")
print(response)

Salida JSON:
```
{
    "fecha": "15 de marzo de 2024",
    "lugar": "Centro de Convenciones de Madrid",
    "precio": 50,
    "descuento_estudiantes": "20%"
}
```


In [14]:
# Parse the JSON
import json

try:
    # Clean the response (remove markdown code blocks if present)
    clean_response = response.strip()
    if clean_response.startswith("```"):
        clean_response = clean_response.split("```")[1]
        if clean_response.startswith("json"):
            clean_response = clean_response[4:]
    
    data = json.loads(clean_response)
    print("JSON parseado correctamente:")
    for key, value in data.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"Error al parsear JSON: {e}")

JSON parseado correctamente:
  fecha: 15 de marzo de 2024
  lugar: Centro de Convenciones de Madrid
  precio: 50
  descuento_estudiantes: 20%


In [15]:
# Markdown table output
prompt_tabla = """Compara los siguientes frameworks de Python para desarrollo web.
Incluye: Django, Flask, FastAPI

Presenta la comparación en formato de tabla Markdown con las siguientes columnas:
- Framework
- Tipo (Full-stack/Micro)
- Curva de aprendizaje
- Mejor para

Incluye solo la tabla, sin texto adicional."""

response = get_completion(prompt_tabla)
print(response)

| Framework | Tipo | Curva de aprendizaje | Mejor para |
| --- | --- | --- | --- |
| Django | Full-stack | Alta | Proyectos complejos y grandes, aplicaciones empresariales |
| Flask | Micro | Baja | Proyectos pequeños y medianos, APIs y servicios web |
| FastAPI | Micro | Media | Desarrollo rápido de APIs y servicios web, proyectos que requieren alta velocidad y rendimiento |


<a name="cot"></a>
## 6. Chain of Thought (CoT)

**Chain of Thought** es una técnica donde pedimos al modelo que "piense paso a paso", mejorando significativamente el razonamiento en problemas complejos.

### Problema matemático

In [16]:
# Without Chain of Thought
problema = """En una tienda, un cliente compra 3 camisetas a 15€ cada una y 2 pantalones a 35€ cada uno.
Tiene un cupón de descuento del 10% sobre el total.
Además, si gasta más de 100€, recibe un descuento adicional de 5€.
¿Cuánto paga el cliente?"""

prompt_sin_cot = f"{problema}\n\nRespuesta:"

response = get_completion(prompt_sin_cot)
print("SIN Chain of Thought:")
print(response)

SIN Chain of Thought:
**Cálculo del total sin descuentos**

* 3 camisetas a 15€ cada una: 3 x 15€ = 45€
* 2 pantalones a 35€ cada uno: 2 x 35€ = 70€
* Total sin descuentos: 45€ + 70€ = 115€

**Aplicación del cupón de descuento del 10%**

* Descuento del 10% sobre el total: 10% de 115€ = 0,10 x 115€ = 11,50€
* Total con descuento del 10%: 115€ - 11,50€ = 103,50€

**Verificación del descuento adicional de 5€**

* El cliente ha gastado más de 100€, por lo que se aplica el descuento adicional de 5€.
* Total final: 103,50€ - 5€ = 98,50€

**Conclusión**

El cliente paga un total de **98,50€**.


In [17]:
# With Chain of Thought
prompt_con_cot = f"{problema}\n\nPiensa paso a paso antes de dar la respuesta final:"

response = get_completion(prompt_con_cot)
print("CON Chain of Thought:")
print(response)

CON Chain of Thought:
**Paso 1: Calcular el costo total de las camisetas**

El cliente compra 3 camisetas a 15€ cada una, por lo que el costo total de las camisetas es:
3 x 15€ = 45€

**Paso 2: Calcular el costo total de los pantalones**

El cliente compra 2 pantalones a 35€ cada uno, por lo que el costo total de los pantalones es:
2 x 35€ = 70€

**Paso 3: Calcular el total de la compra**

El total de la compra es la suma del costo de las camisetas y los pantalones:
45€ + 70€ = 115€

**Paso 4: Aplicar el descuento del 10%**

El cliente tiene un cupón de descuento del 10% sobre el total, por lo que el descuento es:
115€ x 0,10 = 11,50€

El total después del descuento es:
115€ - 11,50€ = 103,50€

**Paso 5: Verificar si se aplica el descuento adicional de 5€**

El cliente gasta más de 100€, por lo que se aplica el descuento adicional de 5€.
El total final es:
103,50€ - 5€ = 98,50€

**Respuesta final**

El cliente paga 98,50€.


### Problema lógico

In [18]:
# Logic puzzle with CoT
puzzle = """Ana es más alta que Beatriz.
Carlos es más bajo que Beatriz.
Diana es más alta que Ana.

¿Quién es el más bajo de los cuatro?

Razona paso a paso:
1. Primero, lista las relaciones de altura
2. Luego, ordena a las personas de más alto a más bajo
3. Finalmente, da la respuesta"""

response = get_completion(puzzle)
print(response)

¡Claro! Empecemos a razonar paso a paso:

**1. Relaciones de altura:**

* Ana es más alta que Beatriz.
* Carlos es más bajo que Beatriz.
* Diana es más alta que Ana.

**2. Orden de altura de más alto a más bajo:**

* Diana es más alta que Ana.
* Ana es más alta que Beatriz.
* Beatriz es más alta que Carlos (porque Carlos es más bajo que Beatriz).

Entonces, el orden de altura de más alto a más bajo es:
1. Diana
2. Ana
3. Beatriz
4. Carlos

**3. La respuesta:**

Basándonos en el orden de altura, podemos concluir que:

El más bajo de los cuatro es **Carlos**.


### Self-Consistency: Múltiples razonamientos

Una técnica avanzada es generar múltiples cadenas de pensamiento y elegir la respuesta más consistente.

In [19]:
# Self-consistency with multiple reasoning paths
problema_logico = """Un granjero tiene que cruzar un río con un lobo, una oveja y un repollo.
Solo puede llevar uno a la vez en su bote.
Si deja al lobo solo con la oveja, el lobo se comerá a la oveja.
Si deja a la oveja sola con el repollo, la oveja se comerá el repollo.

¿Cómo puede el granjero cruzar todo sin que nada sea comido?

Proporciona la solución paso a paso, numerando cada viaje."""

# Generate with higher temperature for diversity
print("Solución al problema del granjero:")
print("=" * 50)
response = get_completion(problema_logico, temperature=0.3)
print(response)

Solución al problema del granjero:
**Solución al problema del granjero y el río**

El granjero necesita cruzar el río con un lobo, una oveja y un repollo, siguiendo las reglas establecidas. A continuación, te presento la solución paso a paso:

1. **Primer viaje:** El granjero lleva a la oveja al otro lado del río y la deja allí. Ahora, el granjero regresa solo al lado inicial.
 * Lado inicial: Granjero, lobo, repollo
 * Lado opuesto: Oveja

2. **Segundo viaje:** El granjero lleva al repollo al otro lado del río y lo deja con la oveja. Sin embargo, no puede dejar a la oveja sola con el repollo, así que regresa con la oveja al lado inicial.
 * Lado inicial: Granjero, lobo, oveja
 * Lado opuesto: Repollo

3. **Tercer viaje:** El granjero lleva al lobo al otro lado del río y lo deja con el repollo. Ahora, el granjero regresa solo al lado inicial.
 * Lado inicial: Granjero, oveja
 * Lado opuesto: Lobo, repollo

4. **Cuarto viaje:** El granjero lleva a la oveja al otro lado del río y la deja

<a name="temperatura"></a>
## 7. Temperatura y Diversidad

El parámetro **temperatura** controla la aleatoriedad de las respuestas:

| Temperatura | Características | Uso recomendado |
|------------|-----------------|----------------|
| 0.0 | Determinístico, consistente | Extracción de datos, código |
| 0.3-0.5 | Balance creatividad/coherencia | Resúmenes, respuestas generales |
| 0.7-1.0 | Creativo, diverso | Escritura creativa, brainstorming |

In [20]:
# Compare different temperatures
prompt_creativo = "Escribe un eslogan publicitario para una cafetería llamada 'El Rincón del Café'."

temperatures = [0.0, 0.5, 1.0]

print("Comparación de temperaturas:")
print("=" * 50)

for temp in temperatures:
    print(f"\nTemperatura {temp}:")
    for i in range(2):  # Generate 2 samples per temperature
        response = get_completion(prompt_creativo, temperature=temp)
        print(f"  {i+1}. {response.strip()}")

Comparación de temperaturas:

Temperatura 0.0:
  1. "Descubre el sabor de la tradición en cada taza, en El Rincón del Café, donde el café es pasión y la hospitalidad es arte."
  2. "Descubre el sabor de la tradición en cada taza, en El Rincón del Café, donde el café es pasión y la hospitalidad es arte."

Temperatura 0.5:
  1. ¡Claro! Aquí te dejo algunas opciones de eslogan publicitario para "El Rincón del Café":

1. **"Disfruta del sabor en cada taza"**: Este eslogan destaca la calidad y el sabor de los cafés que se sirven en la cafetería.
2. **"Un rincón para relajarte y disfrutar"**: Este eslogan enfatiza la atmósfera acogedora y relajante de la cafetería, invitando a los clientes a sentarse y disfrutar de un buen café.
3. **"Donde el café es una experiencia"**: Este eslogan sugiere que la cafetería ofrece más que solo un café, sino una experiencia completa que incluye la atmósfera, el servicio y la calidad de los productos.
4. **"El lugar perfecto para empezar el día"**: Este eslog

<a name="mejores"></a>
## 8. Mejores Prácticas

### DO's y DON'Ts

| ✅ Hacer | ❌ Evitar |
|----------|----------|
| Ser específico y claro | Instrucciones vagas |
| Dar contexto relevante | Asumir que el modelo sabe todo |
| Especificar formato de salida | Dejar el formato abierto |
| Usar delimitadores claros | Mezclar instrucciones y datos |
| Pedir razonamiento paso a paso | Pedir solo la respuesta final |

### Ejemplo de prompt bien estructurado

In [20]:
# Well-structured prompt
prompt_estructurado = """
### ROL ###
Eres un analista de datos senior con experiencia en e-commerce.

### CONTEXTO ###
La empresa vendió 1000 unidades en enero, 1200 en febrero y 800 en marzo.
El precio promedio es de 25€ por unidad.
Los costes fijos mensuales son 5000€.
El margen bruto es del 40%.

### TAREA ###
Analiza las ventas del primer trimestre y proporciona:
1. Ingresos totales del trimestre
2. Beneficio bruto (después de costes fijos)
3. Tendencia de ventas (creciente/decreciente/estable)
4. Una recomendación para Q2

### FORMATO DE SALIDA ###
Presenta los resultados en una lista con viñetas, mostrando los cálculos.
"""

response = get_completion(prompt_estructurado)
print(response)

### Análisis de Ventas del Primer Trimestre

A continuación, se presentan los cálculos y resultados solicitados:

* **Ingresos totales del trimestre**:
 + Enero: 1000 unidades * 25€/unidad = 25.000€
 + Febrero: 1200 unidades * 25€/unidad = 30.000€
 + Marzo: 800 unidades * 25€/unidad = 20.000€
 + Ingresos totales: 25.000€ + 30.000€ + 20.000€ = **75.000€**
* **Beneficio bruto (después de costes fijos)**:
 + Ingresos totales: 75.000€
 + Costes fijos totales (3 meses): 5000€/mes * 3 meses = 15.000€
 + Costes variables (60% del precio de venta, ya que el margen bruto es del 40%): 75.000€ * 0,6 = 45.000€
 + Beneficio bruto: 75.000€ - 45.000€ = **30.000€**
 + Beneficio después de costes fijos: 30.000€ - 15.000€ = **15.000€**
* **Tendencia de ventas**:
 + La tendencia de ventas es **decreciente**, ya que las ventas disminuyeron de 1200 unidades en febrero a 800 unidades en marzo.
* **Recomendación para Q2**:
 + Considerando la tendencia decreciente de ventas, se recomienda **incrementar la pub

<a name="ejercicios"></a>
## 9. Ejercicios Prácticos

### Ejercicio 1: Mejora este prompt

Transforma el prompt ambiguo en uno específico y efectivo.

In [21]:
# Exercise 1: Improve this prompt
prompt_malo = "Escribe sobre inteligencia artificial"

# Your improved prompt:
prompt_mejorado = """
### Escribe tu prompt mejorado aquí ###
# Incluye:
# - Rol específico
# - Contexto del lector objetivo
# - Instrucciones claras sobre longitud, estilo, puntos a cubrir
# - Formato de salida
"""

# Test your improved prompt
# response = get_completion(prompt_mejorado)
# print(response)

### Ejercicio 2: Few-shot para clasificación

Crea un clasificador de tickets de soporte usando few-shot.

In [22]:
# Exercise 2: Support ticket classifier

# Categories: TECHNICAL, BILLING, GENERAL, URGENT

# Provide 2-3 examples for each category, then classify new tickets:
new_tickets = [
    "Mi cuenta fue hackeada y no puedo acceder",
    "¿Cuál es el horario de atención?",
    "Me cobraron dos veces el mismo mes"
]

# Your few-shot prompt:
prompt_clasificador = """
### Escribe tu prompt few-shot aquí ###
# Incluye ejemplos de cada categoría
"""

# for ticket in new_tickets:
#     full_prompt = f"{prompt_clasificador}\n\nTicket: {ticket}\nCategoría:"
#     response = get_completion(full_prompt)
#     print(f"Ticket: {ticket}")
#     print(f"Categoría: {response}\n")

### Ejercicio 3: Chain of Thought para análisis

Usa CoT para analizar pros y contras de una decisión de negocio.

In [23]:
# Exercise 3: Business decision analysis with CoT

decision = """Una startup tecnológica está considerando expandirse a un nuevo mercado (Latinoamérica).
Actualmente opera solo en España con 50 empleados y 2M€ de facturación anual.
Tiene 500K€ disponibles para la expansión."""

# Create a CoT prompt that:
# 1. Analyzes the current situation
# 2. Lists pros and cons
# 3. Considers risks
# 4. Makes a recommendation

prompt_analisis = f"""
### Escribe tu prompt CoT aquí ###

Contexto:
{decision}

Analiza paso a paso:
1. [Análisis de la situación actual]
2. [Pros de la expansión]
3. [Contras y riesgos]
4. [Recomendación final]
"""

# response = get_completion(prompt_analisis)
# print(response)

## Resumen

En este notebook hemos aprendido:

1. **Zero-shot**: Instrucciones claras sin ejemplos
2. **Few-shot**: Proporcionar ejemplos para guiar al modelo
3. **Roles**: Asignar personalidad y expertise al modelo
4. **Formato**: Especificar estructura de salida (JSON, tablas, etc.)
5. **Chain of Thought**: Pedir razonamiento paso a paso
6. **Temperatura**: Controlar creatividad vs. consistencia

### Patrón de prompt efectivo

```
1. ROL: Define quién es el modelo
2. CONTEXTO: Proporciona información relevante
3. TAREA: Instrucciones claras y específicas
4. FORMATO: Cómo quieres la respuesta
5. EJEMPLOS (opcional): Para few-shot
```

En el siguiente notebook veremos cómo construir **chatbots** usando estas técnicas.

---

## Referencias

- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [OpenAI Best Practices](https://platform.openai.com/docs/guides/prompt-engineering)
- [Chain of Thought Paper](https://arxiv.org/abs/2201.11903)
- [Few-Shot Learners Paper](https://arxiv.org/abs/2005.14165)

In [24]:
import session_info
session_info.show(html = False)

-----
ipykernel           7.2.0
langchain_groq      1.1.2
session_info        v1.0.1
-----
IPython             9.10.0
jupyter_client      8.8.0
jupyter_core        5.9.1
-----
Python 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
Windows-11-10.0.26200-SP0
-----
Session information updated at 2026-02-09 15:49
